# Frequentist vs Bayesian Statistics
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout

---
## What you'll explore

By the end of this demo you will be able to:

- **Distinguish** between frequentist and Bayesian interpretations of probability
- **Compare** confidence intervals vs credible intervals on the same problem
- **Identify** which framework is being used when you encounter statistical claims in the real world

> **Tip:** Start with the coin flip widget — the same question, two completely different answers depending on which framework you use.

---
## How we got here

You have already used both frameworks in this series without knowing they had names. In *16: Confidence Intervals* you built ranges around sample estimates, a purely frequentist operation. In *17: Hypothesis Testing* you made binary decisions about whether data was consistent with a null hypothesis, again fully frequentist. In *18: P-Values* you learned that a p-value is always a statement about long-run behavior across hypothetical repeated experiments, never a probability assigned to the hypothesis itself. In *03: Bayes’ Theorem* you saw the opposite move: updating a prior probability with new evidence to get a posterior. This notebook names both frameworks, explains why their answers to the same question look different, and shows you the bigger picture.

---
## Why this matters for data science

Both frameworks appear constantly in machine learning papers, sklearn documentation, and industry practice:

- **Frequentist:** p-values, confidence intervals, A/B testing, hypothesis testing, classical regression inference
- **Bayesian:** Naive Bayes classifier, Bayesian hyperparameter optimization, probabilistic models, credible intervals

Misunderstanding the difference leads to misinterpreting results. A common mistake is saying "there is a 95% probability the true value is in this confidence interval"; this is a Bayesian statement, not a frequentist one. Reading "p < 0.05" in a paper means something quite different from "posterior probability = 0.95," and confusing the two can lead to wrong conclusions.

---
## Try it yourself

In [2]:
from tkh_utils import make_int_slider, make_slider, make_toggle, make_output, display_widget, register_observer

_x_p = np.linspace(0.001, 0.999, 500)

out = make_output()
n_slider          = make_int_slider(1, 200, 1, 20, "Num flips (n):")
k_slider          = make_int_slider(0, 200, 1, 10, "Heads (k):")
framework_toggle  = make_toggle(["Frequentist", "Bayesian"], "Frequentist", "Framework:")
prior_slider      = make_slider(1.0, 10.0, 0.5, 1.0, "Prior strength (α₀ = β₀):")

def render(n, k, framework, prior_strength):
    k = int(np.clip(k, 0, n))

    if framework == "Frequentist":
        p_hat = k / n if n > 0 else 0.5
        se    = np.sqrt(p_hat * (1 - p_hat) / n) if n > 0 and 0 < p_hat < 1 else 1e-6
        ci_lo = max(0.0, p_hat - 1.96 * se)
        ci_hi = min(1.0, p_hat + 1.96 * se)
        z     = (p_hat - 0.5) / se if se > 1e-7 else 0.0
        p_val = float(2 * (1 - stats.norm.cdf(abs(z))))

        x_samp = np.linspace(max(0.001, 0.5 - 5 * se), min(0.999, 0.5 + 5 * se), 500)
        y_samp = stats.norm.pdf(x_samp, 0.5, se)

        x_lo_shade = x_samp[x_samp <= ci_lo]
        x_hi_shade = x_samp[x_samp >= ci_hi]

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x_samp, y=y_samp,
            mode="lines", line=dict(color=PALETTE["primary"], width=3),
            name="Sampling distribution under H₀ (p = 0.5)",
        ))
        for xs in [x_lo_shade, x_hi_shade]:
            if len(xs) > 1:
                fig.add_trace(go.Scatter(
                    x=np.concatenate([[xs[0]], xs, [xs[-1]]]),
                    y=np.concatenate([[0], stats.norm.pdf(xs, 0.5, se), [0]]),
                    fill="toself", fillcolor="rgba(247,103,7,0.30)",
                    line=dict(width=0), showlegend=False,
                ))
        fig.add_vline(x=p_hat, line_color=PALETTE["secondary"],
                      line_width=2.5, line_dash="dash",
                      annotation_text=f"p̂ = {p_hat:.3f}")
        fig.add_vline(x=0.5, line_color=PALETTE["muted"],
                      line_width=1.5, line_dash="dot",
                      annotation_text="H₀: p = 0.5")
        layout = base_layout(
            title=f"Frequentist — p̂ = {p_hat:.3f}, 95% CI = [{ci_lo:.3f}, {ci_hi:.3f}], p = {p_val:.4f}",
            xaxis_title="P(heads)",
            yaxis_title="Probability Density",
        )
        layout.update(xaxis=dict(range=[0, 1]))
        fig.update_layout(layout)

    else:  # Bayesian
        a0     = prior_strength
        b0     = prior_strength
        a_post = a0 + k
        b_post = b0 + (n - k)

        y_prior = stats.beta.pdf(_x_p, a0, b0)
        y_post  = stats.beta.pdf(_x_p, a_post, b_post)
        cri_lo, cri_hi = stats.beta.ppf([0.025, 0.975], a_post, b_post)
        post_mean = a_post / (a_post + b_post)

        x_shade = _x_p[(_x_p >= cri_lo) & (_x_p <= cri_hi)]
        y_shade = stats.beta.pdf(x_shade, a_post, b_post)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=_x_p, y=y_prior,
            mode="lines", line=dict(color=PALETTE["muted"], width=2, dash="dot"),
            name=f"Prior: Beta({a0:.1f}, {b0:.1f})",
        ))
        if len(x_shade) > 1:
            fig.add_trace(go.Scatter(
                x=np.concatenate([[x_shade[0]], x_shade, [x_shade[-1]]]),
                y=np.concatenate([[0], y_shade, [0]]),
                fill="toself", fillcolor="rgba(76,110,245,0.20)",
                line=dict(width=0),
                name=f"95% Credible Interval: [{cri_lo:.3f}, {cri_hi:.3f}]",
            ))
        fig.add_trace(go.Scatter(
            x=_x_p, y=y_post,
            mode="lines", line=dict(color=PALETTE["primary"], width=3),
            name=f"Posterior: Beta({a_post:.1f}, {b_post:.1f})",
        ))
        fig.add_vline(x=0.5, line_color=PALETTE["muted"],
                      line_width=1.5, line_dash="dot",
                      annotation_text="Fair coin (p = 0.5)")
        fig.add_vline(x=post_mean, line_color=PALETTE["accent"],
                      line_width=2, line_dash="dash",
                      annotation_text=f"Posterior mean = {post_mean:.3f}")
        layout = base_layout(
            title=f"Bayesian — Posterior mean = {post_mean:.3f}, 95% Credible Interval = [{cri_lo:.3f}, {cri_hi:.3f}]",
            xaxis_title="P(heads)",
            yaxis_title="Probability Density",
        )
        layout.update(xaxis=dict(range=[0, 1]))
        fig.update_layout(layout)

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

def _on_change(change):
    render(n_slider.value, k_slider.value, framework_toggle.value, prior_slider.value)

register_observer([n_slider, k_slider, framework_toggle, prior_slider], _on_change)
display_widget([n_slider, k_slider, framework_toggle, prior_slider], out)
render(n_slider.value, k_slider.value, framework_toggle.value, prior_slider.value)


---
## What’s happening?

The frequentist and Bayesian frameworks are not just different techniques. They are different answers to the question "what is a probability?"

### Frequentist framework

In the frequentist view, probability is the long-run frequency of an event across many repeated identical experiments. The true parameter (say, the probability a coin lands heads) is a fixed but unknown constant. The data is what is random: it is one sample from the infinite set of samples you could have drawn.

From this view:
- You can never say "the probability the coin is fair is 60%" because the coin either is or is not fair; there is no probability to assign to a fixed truth
- You can say "if the coin were fair, data this extreme would occur only 5% of the time"
- The confidence interval is a statement about a procedure: 95% of intervals built this way capture the true parameter, not a statement about any particular interval

### Bayesian framework

In the Bayesian view, probability is a degree of belief. The parameter is not a fixed unknown; it is a random variable with its own distribution. You start with a prior belief (the prior distribution), observe data, and update your belief using Bayes’ theorem to get the posterior distribution.

From this view:
- You can say "given this data, there is a 72% probability the drug is effective" because probability represents your current state of knowledge
- The credible interval means exactly what it sounds like: there is a 95% probability the parameter is in this range
- Two people starting with different priors will reach different posteriors, even with identical data. This is a feature, not a bug

> **Neither framework is universally correct. Frequentist statistics asks: "Is this data consistent with my hypothesis?" Bayesian statistics asks: "Given this data, what should I believe?" These are different questions and both are valid.**

### Key distinction

| Concept | Frequentist | Bayesian |
|---------|-------------|---------|
| Probability means | Long-run frequency | Degree of belief |
| Parameters are | Fixed, unknown | Random variables |
| Interval means | Procedure captures true value 95% of the time | 95% probability value is in this range |
| Requires prior? | No | Yes |
| p-value | Yes | No equivalent |
| Updates with evidence | No | Yes |

### When data scientists use each

**Frequentist** is the right tool for:
- A/B testing and controlled experiments
- Hypothesis testing in exploratory data analysis
- Classical regression inference (p-values on coefficients)
- Most of what you learned in the statistics section

**Bayesian** is the right tool for:
- The Naive Bayes classifier (prior belief × likelihood = posterior class probability)
- Bayesian hyperparameter optimization (Optuna, scikit-optimize)
- Probabilistic forecasting and uncertainty quantification
- Small sample sizes where a well-reasoned prior improves estimates
- Settings where you want to update beliefs incrementally as data arrives

---
## Real-world example: Drug trial analysis

The same clinical trial data reveals how differently the two frameworks answer the same question. A new drug was tested on 100 patients, 60 improved. We want to know: does this drug work?

The charts below show both analyses side by side. Notice:

- **Notice:** The frequentist p-value (about 0.046) tells you the data is unlikely under a "no effect" assumption, but it does not tell you the probability the drug works
- **Notice:** The Bayesian posterior does give a direct probability that the drug is effective, but it depends on your prior; a skeptical prior pulls the posterior toward 50%, while an optimistic prior pushes it higher
- **Notice:** Both analyses agree the drug likely has an effect, but they answer subtly different questions: one tests whether chance can explain the data, the other directly estimates the probability of efficacy

> **Discussion question:** A pharmaceutical company reports p < 0.05 for their new drug. A Bayesian analysis with a skeptical prior shows only 60% posterior probability of effectiveness. Which result should a doctor trust more? Why?

In [3]:
np.random.seed(42)

# Drug trial: 100 patients, 60 improved
n_patients = 100
n_improved = 60
p_obs      = n_improved / n_patients   # 0.60
p_null     = 0.50                      # H0: no effect beyond chance

# Frequentist analysis
se_null = np.sqrt(p_null * (1 - p_null) / n_patients)
z_obs   = (p_obs - p_null) / se_null
p_value = 2 * (1 - stats.norm.cdf(abs(z_obs)))
z_crit  = stats.norm.ppf(0.975)

se_obs = np.sqrt(p_obs * (1 - p_obs) / n_patients)
ci_lo  = p_obs - z_crit * se_obs
ci_hi  = p_obs + z_crit * se_obs

x_null = np.linspace(0.20, 0.80, 400)
y_null = stats.norm.pdf(x_null, p_null, se_null)

rej_lo = x_null[x_null <= p_null - z_crit * se_null]
rej_hi = x_null[x_null >= p_null + z_crit * se_null]

fig_freq = go.Figure()
fig_freq.add_trace(go.Scatter(
    x=x_null, y=y_null,
    mode="lines",
    line=dict(color=PALETTE["primary"], width=2.5),
    name="Sampling distribution under H₀ (p = 0.50)",
))
for i, rej in enumerate([rej_lo, rej_hi]):
    if len(rej):
        fig_freq.add_trace(go.Scatter(
            x=np.concatenate([[rej[0]], rej, [rej[-1]]]),
            y=np.concatenate([[0], stats.norm.pdf(rej, p_null, se_null), [0]]),
            fill="toself",
            fillcolor="rgba(247,103,7,0.25)",
            line=dict(color=PALETTE["secondary"], width=0),
            name="α = 0.05 rejection region",
            showlegend=(i == 1),
        ))
fig_freq.add_vline(
    x=p_obs,
    line_color=PALETTE["secondary"], line_width=2.5, line_dash="dash",
    annotation_text=f"Observed: {p_obs:.2f}  z = {z_obs:.2f}  p = {p_value:.3f}",
    annotation_position="top left",
)
layout_freq = base_layout(
    title=f"Frequentist: 60/100 patients improved  (z = {z_obs:.2f}, p = {p_value:.3f}, 95% CI: [{ci_lo:.2f}, {ci_hi:.2f}])",
    xaxis_title="Proportion improved",
    yaxis_title="Probability Density under H₀",
)
layout_freq.update(xaxis=dict(range=[0.20, 0.80]))
fig_freq.update_layout(layout_freq)
fig_freq.show()

# Bayesian analysis
# Prior: Beta(10, 10) — previous studies suggest ~50% response rate, moderate certainty
# Posterior: Beta(10+60, 10+40) = Beta(70, 50)
a_prior = 10; b_prior = 10
a_post  = a_prior + n_improved
b_post  = b_prior + (n_patients - n_improved)

x_p     = np.linspace(0.001, 0.999, 500)
y_prior = stats.beta.pdf(x_p, a_prior, b_prior)
y_post  = stats.beta.pdf(x_p, a_post,  b_post)

cred_lo, cred_hi = stats.beta.ppf([0.025, 0.975], a_post, b_post)
post_mean = a_post / (a_post + b_post)

x_shade = x_p[(x_p >= cred_lo) & (x_p <= cred_hi)]
y_shade = stats.beta.pdf(x_shade, a_post, b_post)

fig_bayes = go.Figure()
fig_bayes.add_trace(go.Scatter(
    x=x_p, y=y_prior,
    mode="lines",
    line=dict(color=PALETTE["muted"], width=2, dash="dot"),
    name=f"Prior: Beta({a_prior}, {b_prior}) — from previous studies",
))
fig_bayes.add_trace(go.Scatter(
    x=np.concatenate([[x_shade[0]], x_shade, [x_shade[-1]]]),
    y=np.concatenate([[0], y_shade, [0]]),
    fill="toself",
    fillcolor="rgba(76,110,245,0.20)",
    line=dict(color=PALETTE["primary"], width=0),
    name=f"95% Credible Interval: [{cred_lo:.2f}, {cred_hi:.2f}]",
))
fig_bayes.add_trace(go.Scatter(
    x=x_p, y=y_post,
    mode="lines",
    line=dict(color=PALETTE["primary"], width=2.5),
    name=f"Posterior: Beta({a_post}, {b_post})",
))
fig_bayes.add_vline(
    x=0.5, line_color=PALETTE["muted"], line_width=1.5, line_dash="dot",
    annotation_text="H₀: p = 0.50",
)
fig_bayes.add_vline(
    x=post_mean, line_color=PALETTE["accent"], line_width=2, line_dash="dash",
    annotation_text=f"Posterior mean: {post_mean:.3f}",
    annotation_position="top left",
)
layout_bayes = base_layout(
    title=f"Bayesian: Prior → Posterior update  (posterior mean = {post_mean:.3f})",
    xaxis_title="P(improvement | drug)",
    yaxis_title="Probability Density",
)
layout_bayes.update(xaxis=dict(range=[0.20, 0.90]))
fig_bayes.update_layout(layout_bayes)
fig_bayes.show()

### Comparing the two frameworks

| Framework | What it calculates | What it cannot tell you | When to use it | ML/DS application |
|-----------|-------------------|------------------------|----------------|------------------|
| Frequentist | p-values, confidence intervals, test statistics | Probability that the hypothesis is true | A/B testing, classical inference, hypothesis testing | Regression p-values, t-tests, chi-square tests |
| Bayesian | Posterior distributions, credible intervals, updated beliefs | How often the procedure is correct across repeated samples | Probabilistic models, small samples, sequential updates | Naive Bayes, Bayesian optimization, probabilistic forecasting |

---
## Key takeaway

> **Frequentist statistics asks whether your data is consistent with a hypothesis; Bayesian statistics asks what you should believe given your data — and knowing which question you’re answering is as important as the answer itself.**

---
*Next up: You’ve completed the statistics working sessions — head to math/ to see the mathematical foundations that underpin everything you just learned*